## Example 1: Key constants in readchar.key

In [ ]:
# Example 1: Key constants in readchar.key
import os

import readchar
from readchar import key

# readkey() returns these strings, so a program compares its result with them.
assert key.CTRL_C == "\x03"
assert key.ESC == "\x1b"
assert key.SPACE == " "
assert key.TAB == "\t"
assert key.ENTER in ("\n", "\r")  # "\r" on Windows
assert key.BACKSPACE in ("\x7f", "\x08")  # "\x08" on Windows
if os.name == "posix":
    # The ANSI escape sequences a terminal sends for the arrow keys
    assert key.UP == "\x1b[A"
    assert key.DOWN == "\x1b[B"
    assert key.RIGHT == "\x1b[C"
    assert key.LEFT == "\x1b[D"
assert callable(readchar.readchar) and callable(readchar.readkey)

for name in ("UP", "DOWN", "LEFT", "RIGHT", "ENTER", "ESC", "BACKSPACE", "CTRL_C"):
    print(f"{name:9} {getattr(key, name)!r}")

## Example 2: Interrupt keys in readchar.config

In [ ]:
# Example 2: Interrupt keys in readchar.config
import readchar
from readchar import config, key

# readkey() raises KeyboardInterrupt when it reads one of these keys.
assert readchar.config is config
assert config.INTERRUPT_KEYS == [key.CTRL_C], config.INTERRUPT_KEYS

# config is a holder for class attributes and cannot be instantiated.
try:
    config()
except SyntaxError as error:
    print("config() ->", error)
else:
    raise AssertionError("readchar.config() should refuse to create an instance")
print("INTERRUPT_KEYS:", config.INTERRUPT_KEYS)

## Example 3: readchar() reads one character from a terminal

The cell runs a child process whose stdin is a pseudo-terminal and types into it.

In [ ]:
# Example 3: readchar() reads one character from a terminal
# readchar reads from a real terminal. This cell starts a child Python process whose stdin is a
# pseudo-terminal and types into it. Pseudo-terminals exist only on POSIX systems.
import os
import subprocess
import sys
import time


def type_into_child(code, keystrokes, timeout=15.0):
    """Run `code` with a pseudo-terminal as stdin, type each keystroke, and return one stdout line per keystroke."""
    import pty
    import select
    import tty

    master, slave = pty.openpty()
    tty.setraw(slave)
    child = subprocess.Popen([sys.executable, "-c", code], stdin=slave,
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    os.close(slave)
    try:
        lines = []
        for data in keystrokes:
            # readchar discards pending input when it switches the terminal mode, so type the
            # keystroke again until the child reports what it read.
            deadline = time.monotonic() + timeout
            while True:
                os.write(master, data)
                ready, _, _ = select.select([child.stdout], [], [], 0.2)
                if ready:
                    lines.append(child.stdout.readline().decode().rstrip("\n"))
                    break
                if child.poll() is not None or time.monotonic() > deadline:
                    raise AssertionError(f"child read nothing: {child.stderr.read().decode()}")
        assert child.wait(timeout=timeout) == 0, child.stderr.read().decode()
        return lines
    finally:
        if child.poll() is None:
            child.kill()
        os.close(master)


if os.name == "posix":
    lines = type_into_child("import readchar\nprint(repr(readchar.readchar()), flush=True)", [b"a"])
    assert lines == ["'a'"], lines
    print("readchar() returned", lines[0])
else:
    print("Skipped: this cell needs a POSIX pseudo-terminal")

## Example 4: readkey() turns an escape sequence into one key

In [ ]:
# Example 4: readkey() turns an escape sequence into one key
# Uses type_into_child() from Example 3.
import os

from readchar import key

if os.name == "posix":
    script = (
        "import readchar\n"
        "print(repr(readchar.readkey()), flush=True)\n"
        "print(repr(readchar.readkey()), flush=True)\n"
    )
    lines = type_into_child(script, [b"\x1b[A", b"x"])
    assert lines == [repr(key.UP), "'x'"], lines
    print("arrow key ->", lines[0], "| plain key ->", lines[1])
else:
    print("Skipped: this cell needs a POSIX pseudo-terminal")

## Example 5: Ctrl+C and INTERRUPT_KEYS

In [ ]:
# Example 5: Ctrl+C raises KeyboardInterrupt unless INTERRUPT_KEYS is changed
# Uses type_into_child() from Example 3.
import os

from readchar import key

if os.name == "posix":
    default = (
        "import readchar\n"
        "try:\n"
        "    readchar.readkey()\n"
        "except KeyboardInterrupt:\n"
        "    print('KeyboardInterrupt', flush=True)\n"
    )
    assert type_into_child(default, [b"\x03"]) == ["KeyboardInterrupt"]

    # With no interrupt keys, readkey() returns Ctrl+C like any other key.
    no_interrupt = (
        "import readchar\n"
        "readchar.config.INTERRUPT_KEYS = []\n"
        "print(repr(readchar.readkey()), flush=True)\n"
    )
    lines = type_into_child(no_interrupt, [b"\x03"])
    assert lines == [repr(key.CTRL_C)], lines
    print("default: KeyboardInterrupt | INTERRUPT_KEYS = []:", lines[0])
else:
    print("Skipped: this cell needs a POSIX pseudo-terminal")